# §36: **L0dfi** = the LaGoVAD trunk (CLIP only) trained on K5dfi's recipe and corpus, read zero-shot on whole DoTA-CAP and DADA-2000

Rule: `core/docs/final/KIM_RUN.md` **§36**, written before any number. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first. **PRELIMINARY (n = 1, s2024); printed, decides nothing; K5f stays the reference and the main model.** `nexar_test` is never touched.

**L0dfi = K5dfi's training command with `kim.enabled=false`, `v2.motion=none`, `model.motion_dim=0`** (`kim.prototypes` / `kim.edge_mask` / `kim.seam_mask` back to their defaults: the config refuses them without KIM). Same corpus (`Nexar_window`, §31 labels), same DVS (θ 0.7 / 0.85, `ignore`), same steps (1,749), same seed. Input = the plain CLIP caches (A0's inputs): `Nexar_s8_ncc` to train, `DoTA_s1_ncc[::3]` and `DADA2000_orig` to read. Step 5 asserts those plain rows equal the CLIP columns of the bakes K5dfi trained and read on, so every row sees the same CLIP frames.

| row | what | here |
|---|---|---|
| **K5dfi** | control (§34): VideoMAE V2-S motion + KIM, same recipe | reads only (`kim_cap_eval` §35, `kim_dvs_ignore`) |
| **L0dfi** | K5dfi without motion and KIM | **train + read** |
| K5f | reference (§31.1) | reads only |
| A0 (v2 Nexar) | LaGoVAD trunk, DVS `span`, pre-§31 labels, 5-seed average | printed if its reads are on Drive |
| LaGoVAD | released checkpoint, protocol B | reads only (§35, `zs_lagovad`) |

| step | what | GPU | estimate |
|---|---|---|---|
| 1-3 | setup, deps, preflight (K5dfi run + reads, caches, census) | no | 2-5 min |
| 4 | pytest on the touched code -- **raises if red** | no | 1-2 min |
| 5 | corpus check + **same CLIP frames** (`check-inputs`: Nexar train 900, DADA 370, DoTA-CAP 569 + 560) | no | 5-15 min |
| 6 | train **L0dfi** (skips if verified on Drive); J10, config diff to K5dfi, `mul_mil` floor | **yes** | <= K5dfi's time |
| 7 | read L0dfi: DoTA-CAP-dev, DoTA-CAP-eval (`--final`, marker), pool -> whole DoTA-CAP; DADA whole | **yes** | 15-30 min |
| 8 | `zs_arm_table` on whole DoTA-CAP + DADA (control K5dfi) -> **paste `REPORTS/zs_arm_table_l0dfi_s2024.md` to Claude** | no | 5-10 min |

Everything new goes to `Thesis-Final/outputs/kim_lagovad_nexar/`. A re-run skips what is on Drive. **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import json
import os
import pickle
import shutil
import subprocess
import sys
import time
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only: data, cache/clip (DoTA_s1_ncc, DADA2000_orig), ckpts
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window, Nexar caches, A2 bakes, KNN, census, v2 A0 reads
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, K5dfi / K5f / LaGoVAD reads; every output of this notebook
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

os.environ['REPO'] = f'{DRIVE_F}/kat-vad'
REPO = Path(os.environ['REPO'])
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert '## 36. L0dfi' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), 'KIM_RUN.md has no §36 -- re-upload kat-vad/'
assert (REPO / 'core/tools/dota_cap_pool.py').is_file(), 'no dota_cap_pool -- re-upload kat-vad/'
assert 'if not is_abnormal:\n            return NORMAL_CLASS' in (REPO / 'core/data/dataset.py').read_text(), \
    'core/data/dataset.py has no §31 fix -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

K5       = constants.KIM_MAIN_ARM                       # "K5": names run / eval sub-folders only
K5F      = constants.KIM_REFERENCE_ARM
K5DFI    = constants.KIM_DVS_IGNORE_ROW                 # the control
L0       = constants.KIM_LAGOVAD_NEXAR_ROW              # 'L0dfi'
LAG      = constants.ZS_LAGOVAD_ARM
A0       = 'A0'
SEED     = constants.KIM_SEED                           # 2024
OFF      = constants.V2_OFF
DATASET  = constants.NEXAR_DATASET
C_WINDOW = constants.NEXAR_CONSTRUCTION_WINDOW
T2NAME   = constants.DADA_ORIGIN_DATASET
CAP_DEV, CAP_EVAL = constants.V2_SPLIT_DOTA_CAP_DEV, constants.V2_SPLIT_DOTA_CAP_EVAL
CAP_ALL  = constants.DOTA_CAP_ALL_DIRNAME
STRIDE   = constants.V2_E1_STRIDE_BC                    # protocol B: s1[::3]
N_DEV, N_EVAL = constants.DOTA_CAP_DEV_CLIPS, constants.DOTA_CAP_EVAL_CLIPS

OUT_F    = constants.OUTPUT_ROOT
V2_CACHE, V2_OUT = Path(DRIVE_V2) / 'cache', Path(DRIVE_V2) / 'outputs'
WINDOW   = Path(DRIVE_V2) / 'data' / f'{DATASET}_{C_WINDOW}'         # K5dfi's corpus (read-only)
KNN      = V2_CACHE / 'knn' / f'{DATASET}_{C_WINDOW}' / constants.KNN_CACHE_FILENAME
CENSUS   = V2_OUT / 'v2' / 'REPORTS' / 'nexar_n0' / 'census.json'
# K5dfi: the control (§34 run + DADA read; §35 whole DoTA-CAP read)
K5DFI_RUN  = OUT_F / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFI / K5 / f's{SEED}'
CAP_EVAL_OUT = OUT_F / 'kim_cap_eval'                                  # §35's folder
# plain CLIP caches = L0dfi's inputs; the A2 bakes = K5dfi's inputs (CLIP columns first)
CLIP_S8  = V2_CACHE / 'clip' / 'Nexar_s8_ncc'
DOTA_S1  = Path(DRIVE) / 'cache' / 'clip' / 'DoTA_s1_ncc'
T2_CLIP  = Path(DRIVE) / 'cache' / 'clip' / T2NAME
A2       = V2_CACHE / 'v2' / 'A2_S'
A2_NEXAR = A2 / 'Nexar_s8'
A2_DADA  = A2 / f'{T2NAME}_test_sources_nexar'
A2_CAP   = {CAP_DEV: A2 / f'{CAP_DEV}_s3_from_s1_nexar', CAP_EVAL: A2 / f'{CAP_EVAL}_s3_from_s1_nexar'}
T2       = constants.DATA_ROOT / T2NAME
T2_CENSUS = T2 / 'counts' / 'resolved_census.json'
ANNOT    = constants.DATA_ROOT / 'DADA2000Origin' / constants.DADA_ORIGIN_ROOT_DIRNAME / constants.DADA_ORIGIN_ANNOTATION_FILENAME
DOTA     = constants.DATA_ROOT / 'DoTA'

ROOT     = OUT_F / constants.KIM_LAGOVAD_NEXAR_OUTPUT_DIRNAME          # this notebook
RUN_DIR  = ROOT / L0 / K5 / f's{SEED}'
CKPT     = RUN_DIR / 'checkpoint_last.pt'
EVAL     = ROOT / L0 / 'eval'
REPORTS  = ROOT / 'REPORTS'
LOCAL    = Path('/content/l0dfi')
LOCAL.mkdir(parents=True, exist_ok=True)

# the comparison rows' reads (dir per set); A0 is optional (printed if on Drive)
A0_EVAL  = V2_OUT / 'v2_nexar_window' / 'eval'
READS = {
    K5DFI: {'dota': CAP_EVAL_OUT / K5DFI / 'eval' / CAP_ALL / K5,
            'dada': OUT_F / constants.KIM_DVS_IGNORE_OUTPUT_DIRNAME / K5DFI / 'eval' / 'dada' / K5},
    L0:    {'dota': EVAL / CAP_ALL / K5, 'dada': EVAL / 'dada' / K5},
    K5F:   {'dota': CAP_EVAL_OUT / K5F / 'eval' / CAP_ALL / K5,
            'dada': OUT_F / constants.KIM_FIXED_OUTPUT_DIRNAME / K5F / 'eval' / 'dada' / K5},
    LAG:   {'dota': CAP_EVAL_OUT / LAG / 'eval' / CAP_ALL / 'B',
            'dada': OUT_F / constants.ZS_LAGOVAD_OUTPUT_DIRNAME / 'eval' / 'dada' / LAG},
}
INPUTS = (('Nexar_window', WINDOW / constants.LABELS_TRAIN_FILENAME), ('KNN cache', KNN), ('Nexar census', CENSUS),
          ('K5dfi checkpoint', K5DFI_RUN / 'checkpoint_last.pt'), ('K5dfi verified', K5DFI_RUN / 'verified.json'),
          ('K5dfi config', K5DFI_RUN / 'config.yaml'), ('K5dfi metrics', K5DFI_RUN / 'metrics.jsonl'),
          ('Nexar_s8_ncc', CLIP_S8), ('DoTA s1', DOTA_S1), ('T2 CLIP', T2_CLIP),
          ('A2 Nexar bake', A2_NEXAR), ('A2 DADA bake', A2_DADA), ('A2 CAP-dev bake', A2_CAP[CAP_DEV]),
          ('A2 CAP-eval bake', A2_CAP[CAP_EVAL]), ('T2 meta', T2 / constants.META_FILENAME), ('T2 census', T2_CENSUS),
          ('DADA annotation', ANNOT), ('DoTA metadata', DOTA / 'metadata_val.json'),
          *[(f'{row} {s} read', d / ('protocol_b_readout.json' if s == 'dota' else 'dada_readout.json'))
            for row, by in READS.items() if row != L0 for s, d in by.items()])
missing = [f'{n}: {p}' for n, p in INPUTS if not p.exists()]
assert not missing, 'missing -- paste to Claude:\n' + '\n'.join(missing)
print(f'{len(INPUTS)} inputs found | runs / reads -> {ROOT}')

In [ ]:
%%bash
pip install -q "transformers==4.56.*" av einops faiss-cpu scikit-learn openpyxl xlrd
python -c "import av, einops, faiss, sklearn, openpyxl; print('deps OK')"
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU -- steps 6-7 need one'
df -h /content | tail -1

In [ ]:
import numpy as np
import torch
import yaml

from core.config import Config                       # noqa: E402
from core.data.v2_splits import load_split           # noqa: E402
from core.tools.dada_whole_eval import test_sources  # noqa: E402
from core.tools.v5_stage1 import _flat               # noqa: E402

ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')   # transformers must not import TF


def run(cmd, log=None):
    '''Stream a child's output here (and to ``log``); raise with its tail on failure.'''
    tail = []
    sink = open(log, 'a') if log else None
    try:
        with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
            for line in proc.stdout:
                tail = (tail + [line])[-40:]
                if sink:
                    sink.write(line)
                print(line, end='')
    finally:
        if sink:
            sink.close()
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


def stage(src, dst, ids=None):
    '''Copy a Drive dir (only ``ids`` + json/npz manifests if given) or file to VM disk once (bit-identical; C2).'''
    t0 = time.time()
    if src.is_file():
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
        return dst
    dst.mkdir(parents=True, exist_ok=True)
    if ids is None:
        for f in src.iterdir():
            if f.is_file() and not (dst / f.name).exists():
                shutil.copy2(f, dst / f.name)
    else:
        for v in ids:
            if not (dst / f'{v}.npy').exists():
                shutil.copy2(src / f'{v}.npy', dst / f'{v}.npy')
        for pattern in ('*.json', '*.npz'):
            for m in src.glob(pattern):
                shutil.copy2(m, dst / m.name)
    print(f'  staged {src.name} in {time.time() - t0:.0f}s')
    return dst


def publish(src, dst):
    '''VM-local result -> Drive ATOMICALLY: ``<dst>.partial``, fsync, rename (C10).'''
    if src.is_dir():
        for f in [p for p in src.rglob('*') if p.is_file()]:
            publish(f, dst / f.relative_to(src))
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    tmp = dst.with_name(dst.name + '.partial')
    shutil.copy2(src, tmp)
    with open(tmp, 'rb+') as fh:
        os.fsync(fh.fileno())
    os.replace(tmp, dst)


def flush_drive():
    drive.flush_and_unmount()
    drive.mount('/content/drive')


def metrics_rows(path):
    return [json.loads(l) for l in path.read_text().splitlines() if l.strip()] if path.exists() else []


def ckpt_step(path):
    '''The checkpoint's own global_step; 0 if absent, -1 if unreadable (never equals a real step).'''
    if not path.exists():
        return 0
    try:
        return int(torch.load(path, map_location='cpu', weights_only=False).get('global_step', 0))
    except (RuntimeError, EOFError, pickle.UnpicklingError) as exc:
        print(f'UNREADABLE {path}: {exc}')
        return -1


def is_verified(run_dir, steps):
    rec, ckpt = run_dir / 'verified.json', run_dir / 'checkpoint_last.pt'
    if rec.exists() and ckpt.exists():
        r = json.loads(rec.read_text())
        return r['step'] == steps and r['ckpt_bytes'] == ckpt.stat().st_size
    return False


def write_ids(path, ids):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(''.join(f'{v}\n' for v in ids))
    return path


# Preflight on small files only (C39): K5dfi is the recipe
dfi_cfg = _flat(yaml.safe_load((K5DFI_RUN / 'config.yaml').read_text()))
assert dfi_cfg['kim.enabled'] and dfi_cfg['v2.motion'] == constants.V5_ENCODER, 'K5DFI_RUN is not K5dfi'
assert dfi_cfg['loss.dvs_anchor_mode'] == constants.DVS_ANCHOR_MODE_IGNORE and dfi_cfg['dvs.theta'] == 0.7, 'not K5dfi'
assert dfi_cfg['data.dataset'] == DATASET and dfi_cfg['train.seed'] == SEED and not dfi_cfg['data.is_egocentric']
STEPS = json.loads((K5DFI_RUN / 'verified.json').read_text())['step']             # 1,749
EPOCHS = dfi_cfg['train.num_epochs']                                              # 33
assert is_verified(K5DFI_RUN, STEPS) and metrics_rows(K5DFI_RUN / 'metrics.jsonl')[-1]['global_step'] == STEPS, 'K5dfi: J10'
assert (CLIP_S8 / constants.V2_INPUT_MANIFEST_FILENAME).exists() is False, 'Nexar_s8_ncc carries a v2 manifest: not a plain CLIP cache'
census = json.loads(CENSUS.read_text())['videos']
TRAIN_IDS = sorted(load_split(constants.V2_SPLIT_NEXAR_TRAIN))
DADA_IDS = test_sources(T2)
CAP_IDS = {CAP_DEV: sorted(load_split(CAP_DEV)), CAP_EVAL: sorted(load_split(CAP_EVAL, final=True))}   # ids only (§36)
assert len(TRAIN_IDS) == 900 and len(DADA_IDS) == 370 and (len(CAP_IDS[CAP_DEV]), len(CAP_IDS[CAP_EVAL])) == (N_DEV, N_EVAL)
for name, d, ids in (('Nexar_s8_ncc', CLIP_S8, TRAIN_IDS), ('T2 CLIP', T2_CLIP, DADA_IDS),
                     ('DoTA s1', DOTA_S1, CAP_IDS[CAP_DEV] + CAP_IDS[CAP_EVAL])):
    miss = [v for v in ids if not (d / f'{v}.npy').exists()]
    assert not miss, f'{len(miss)} of {name} missing: {miss[:5]}'
print(f'preflight OK: K5dfi step {STEPS} ({EPOCHS} epochs) | train {len(TRAIN_IDS)} | DoTA-CAP {N_DEV} + {N_EVAL} | DADA {len(DADA_IDS)}')

## 4. Tests on the touched code (CPU) -- **raises if anything fails: stop and paste the output**

In [ ]:
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_dota_cap_pool.py',
     'core/tests/test_baseline_losses.py', 'core/tests/test_dvs_dataset.py'])

## 5. K5dfi's corpus, and the same CLIP frames for every row (`zs_lagovad_readout check-inputs`, §36 R4)

The plain caches L0dfi reads must equal the first 512 (CLIP) columns of the bakes K5dfi read, on every item. Each bake is staged, checked, then deleted from the VM.

In [ ]:
labels = json.loads((WINDOW / constants.LABELS_TRAIN_FILENAME).read_text())
n_pos = sum(labels.values())
assert (len(labels), n_pos) == (7311, 1689), f'not K5dfi\'s corpus: {len(labels)} windows, {n_pos} positive'
steps_per_epoch = -(-2 * n_pos // 64)            # balanced epoch = 2 x positives, batch 64
assert steps_per_epoch * EPOCHS == STEPS, f'{steps_per_epoch} x {EPOCHS} != K5dfi {STEPS}'
print(f'Nexar_window: {n_pos} positive / {len(labels)} windows | {steps_per_epoch} steps/epoch x {EPOCHS} = {STEPS}')

L_PLAIN = {'nexar': stage(CLIP_S8, LOCAL / 'stage' / 'nexar_s8', TRAIN_IDS),
           'dada': stage(T2_CLIP, LOCAL / 'stage' / 'dada_clip', DADA_IDS),
           'dota': stage(DOTA_S1, LOCAL / 'stage' / 'dota_s1', CAP_IDS[CAP_DEV] + CAP_IDS[CAP_EVAL])}
CHECKS = (('nexar', A2_NEXAR, TRAIN_IDS, 1), ('dada', A2_DADA, DADA_IDS, 1),
          ('dota', A2_CAP[CAP_DEV], CAP_IDS[CAP_DEV], STRIDE), ('dota', A2_CAP[CAP_EVAL], CAP_IDS[CAP_EVAL], STRIDE))
for plain, bake, ids, stride in CHECKS:
    done = LOCAL / 'checks' / f'{bake.name}.ok'
    if done.exists():
        print(f'{bake.name}: checked earlier this session')
        continue
    l_bake = stage(bake, LOCAL / 'stage' / 'bake_check' / bake.name, ids)
    run([sys.executable, '-m', 'core.tools.zs_lagovad_readout', 'check-inputs', '--plain', L_PLAIN[plain],
         '--bake', l_bake, '--ids-file', write_ids(LOCAL / 'ids' / f'{bake.name}.txt', ids), '--plain-stride', stride])
    shutil.rmtree(l_bake)
    done.parent.mkdir(parents=True, exist_ok=True)
    done.write_text('ok')
print('same CLIP frames: Nexar train, DADA, DoTA-CAP dev + eval')

## 6. Train **L0dfi** (§36 R1)

K5dfi's `--set` list with `kim.enabled=false`, `v2.motion=none`, `model.motion_dim=0` (no `kim.*` sub-flags), on `Nexar_s8_ncc`. Checks: J10, the config diff to K5dfi is exactly the six keys of R4, last-epoch `mul_mil` > the §31 floor. **Watch the log: past 2 s/step, stop and paste it.**

In [ ]:
SETS = ['train.stage=2', f'train.seed={SEED}', f'train.num_epochs={EPOCHS}', 'train.amp=true',
        f'data.dataset={DATASET}', f'loss.mil_topk_pct={constants.DADA_ORIGIN_MIL_TOPK_PCT}',
        'kip.enabled=false', f'loss.dvs_anchor_mode={constants.DVS_ANCHOR_MODE_IGNORE}', 'data.is_egocentric=false',
        f'model.score_head_kernel={constants.DADA_ORIGIN_SCORE_HEAD_KERNEL}', 'dvs.theta=0.7', 'dvs.theta_ego=0.85',
        'kim.enabled=false', f'v2.motion={OFF}', 'model.motion_dim=0']
WANT_DIFF = {'kim.enabled', 'kim.prototypes', 'kim.edge_mask', 'kim.seam_mask', 'v2.motion', 'model.motion_dim'}
DEFAULTS = _flat(Config().to_dict())


def config_diff(base, got):
    '''Keys that differ; a key added after ``base`` trained may sit at its default.'''
    out = {}
    for k in set(base) | set(got):
        if base.get(k) == got.get(k):
            continue
        if k not in base and got.get(k) == DEFAULTS.get(k):
            continue
        if k not in got and base.get(k) == DEFAULTS.get(k):
            continue
        out[k] = (base.get(k), got.get(k))
    return out


def check_run():
    rows = metrics_rows(RUN_DIR / 'metrics.jsonl')
    assert ckpt_step(CKPT) == rows[-1]['global_step'] == STEPS, f'{L0}: J10'
    got = _flat(yaml.safe_load((RUN_DIR / 'config.yaml').read_text()))
    diff = config_diff(dfi_cfg, got)
    assert set(diff) == WANT_DIFF, f'{L0} must differ from K5dfi in exactly {sorted(WANT_DIFF)}: {diff}'
    assert not got['kim.enabled'] and got['v2.motion'] == OFF and got['model.motion_dim'] == 0, diff
    last = [r for r in rows if r['epoch'] == rows[-1]['epoch']]
    mul_last = sum(r['mul_mil'] for r in last) / len(last)
    assert mul_last > constants.KIM_FIXED_MUL_MIL_FLOOR, f'{L0}: last-epoch mul_mil {mul_last:.4f} -- pre-§31 code'
    ref = metrics_rows(K5DFI_RUN / 'metrics.jsonl')[0]
    print(f'{L0} config diff to K5dfi: {diff}\n  last-epoch mul_mil {mul_last:.4f} > {constants.KIM_FIXED_MUL_MIL_FLOOR}')
    print(f"  step 1 (printed only, no matched start): mil {rows[0]['mil']:.6f} (K5dfi {ref['mil']:.6f}) | "
          f"dvs_sup {rows[0]['dvs_sup']:.6f} (K5dfi {ref['dvs_sup']:.6f})")


if is_verified(RUN_DIR, STEPS):
    print(f'{L0} s{SEED}: verified on Drive -- not retrained')
else:
    assert torch.cuda.is_available(), 'torch sees no GPU -- train.device=auto would silently train on CPU'
    local = LOCAL / 'runs' / L0
    local.mkdir(parents=True, exist_ok=True)
    resume = ['--resume', local / 'checkpoint_last.pt'] if (local / 'checkpoint_last.pt').exists() else []
    t0 = time.time()
    run([sys.executable, '-m', 'core.train', *[x for s in SETS for x in ('--set', s)],
         '--data-dir', stage(WINDOW, LOCAL / 'stage' / 'data'), '--clip-dir', L_PLAIN['nexar'],
         '--output-dir', local, '--knn-cache', stage(KNN, LOCAL / 'stage' / 'knn' / KNN.name), *resume],
        log=LOCAL / f'train_{L0}.log')
    last = metrics_rows(local / 'metrics.jsonl')[-1]
    assert ckpt_step(local / 'checkpoint_last.pt') == last['global_step'] == STEPS, f'{L0}: J10 locally {last["global_step"]}'
    for name in ('checkpoint_last.pt', 'metrics.jsonl', 'config.yaml'):
        publish(local / name, RUN_DIR / name)
    flush_drive()
    assert ckpt_step(CKPT) == STEPS, f'{L0}: Drive copy is not step {STEPS}'
    (local / 'verified.json').write_text(json.dumps({'step': STEPS, 'ckpt_bytes': CKPT.stat().st_size}))
    publish(local / 'verified.json', RUN_DIR / 'verified.json')
    print(f'{L0}: trained in {(time.time() - t0) / 60:.0f} min | mil {last.get("mil")}')
check_run()

## 7. Read L0dfi: DoTA-CAP-dev, DoTA-CAP-eval (`--final`, §36), pool -> whole DoTA-CAP; DADA whole

Same calls as v2's A0 reads (`nexar_zeroshot.ipynb`): plain `DoTA_s1_ncc` at `s1[::3]`, plain `DADA2000_orig`, the Nexar definitions. The eval half is never printed alone.

In [ ]:
from core.tools import dota_cap_pool  # noqa: E402

MARK = ROOT / 'OPENED.json'
if not MARK.exists():
    ROOT.mkdir(parents=True, exist_ok=True)
    MARK.write_text(json.dumps({'opened_utc': time.strftime('%Y-%m-%dT%H:%MZ', time.gmtime()), 'split': CAP_EVAL,
                                'rows': [L0], 'seed': SEED, 'authorized': 'user, 2026-11-21',
                                'rule': 'KIM_RUN.md §36: printed, decides nothing',
                                'checkpoint': {'path': str(CKPT), 'bytes': CKPT.stat().st_size}}, indent=2))
L_LAB = stage(DOTA / 'labels_s8', LOCAL / 'stage' / 'dota_labels')
for split in (CAP_DEV, CAP_EVAL):
    out = EVAL / split / K5
    if not (out / 'protocol_b_readout.json').exists():
        local = LOCAL / 'eval' / split
        shutil.rmtree(local, ignore_errors=True)
        final = ['--final'] if split == CAP_EVAL else []
        run([sys.executable, '-m', 'core.tools.protocol_b_eval', '--run', f's{SEED}', CKPT, *final,
             '--input-dir', L_PLAIN['dota'], '--s1-dir', L_PLAIN['dota'], '--metadata', DOTA / 'metadata_val.json',
             '--split-file', DOTA / 'val_split.txt', '--data-dir', L_LAB, '--split', split, '--stride', STRIDE,
             '--out-dir', local, '--device', 'auto'], log=LOCAL / f'{L0}_{split}.log')
        publish(local, out)
    r = json.loads((out / 'protocol_b_readout.json').read_text())
    assert r['split'] == split and r['clips'] == len(CAP_IDS[split]) and r['global_steps'] == {f's{SEED}': STEPS}, r
    assert (r['arm']['crn'], r['arm']['motion']) == (OFF, OFF), r['arm']
    print(f'{L0} {split}: read ({r["clips"]} clips)')

out = READS[L0]['dota']
if not (out / 'protocol_b_readout.json').exists():
    local = LOCAL / 'eval' / CAP_ALL
    shutil.rmtree(local, ignore_errors=True)
    dota_cap_pool.write_pooled(EVAL / CAP_DEV / K5, EVAL / CAP_EVAL / K5, local, (N_DEV, N_EVAL))
    publish(local, out)
print(f'{L0} whole DoTA-CAP: {json.loads((out / "protocol_b_readout.json").read_text())["clips"]} clips pooled')

L_T2 = LOCAL / 'T2_meta'
L_T2.mkdir(parents=True, exist_ok=True)
shutil.copy2(T2 / constants.META_FILENAME, L_T2 / constants.META_FILENAME)
L_DEFS = LOCAL / 'nexar_defs'
L_DEFS.mkdir(parents=True, exist_ok=True)
for f in WINDOW.glob('*.json'):             # the definitions every Nexar row read with
    shutil.copy2(f, L_DEFS / f.name)
out = READS[L0]['dada']
if not (out / 'clip_aucs.json').exists():
    local = LOCAL / 'eval' / 'dada'
    shutil.rmtree(local, ignore_errors=True)
    run([sys.executable, '-m', 'core.tools.dada_whole_eval', '--run', f's{SEED}', CKPT, '--t2-dir', L_T2,
         '--annotation', ANNOT, '--census', T2_CENSUS, '--input-dir', L_PLAIN['dada'],
         '--data-dir', L_DEFS, '--out-dir', local], log=LOCAL / f'{L0}_dada.log')
    publish(local, out)
r = json.loads((out / 'dada_readout.json').read_text())
assert r['videos'] == len(DADA_IDS) and r['global_steps'] == {f's{SEED}': STEPS} and r['arm']['motion'] == OFF, r
print(f'{L0} DADA whole: read ({r["videos"]} sources)')
flush_drive()

## 8. `zs_arm_table`: whole DoTA-CAP (1,129) + DADA-2000 (370), control **K5dfi** -> paste `REPORTS/zs_arm_table_l0dfi_s2024.md` to Claude

Rows K5dfi, L0dfi, K5f, LaGoVAD (public ckpt), and v2's Nexar A0 when its reads are on Drive (pooled here the same way). `L0dfi - K5dfi` is printed as arm − control: flip the sign for "the model's effect". DADA's middle ruler (0.7342) is printed from the read-out. Printed only; it decides nothing.

In [ ]:
from core.tools import zs_arm_table as zat  # noqa: E402

a0_dev, a0_eval, a0_dada = (A0_EVAL / 'dota' / A0 / CAP_DEV, A0_EVAL / 'dota' / A0 / CAP_EVAL, A0_EVAL / 'dada' / A0)
if all((d / 'clip_scores.npz').exists() for d in (a0_dev, a0_eval, a0_dada)):
    a0_all = LOCAL / 'a0' / CAP_ALL
    if not (a0_all / 'protocol_b_readout.json').exists():
        dota_cap_pool.write_pooled(a0_dev, a0_eval, a0_all, (N_DEV, N_EVAL))
    READS[A0] = {'dota': a0_all, 'dada': a0_dada}
else:
    print(f'v2 Nexar A0 reads not found under {A0_EVAL}: its row is left out')

NAME = f'l0dfi_s{SEED}'
local = LOCAL / 'report'
old_pool, old_name = zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA]
zat.POOLING[zat.DOTA] = f'{N_DEV + N_EVAL} clips, protocol B; micro / AP after per-clip min-max'
zat.SET_NAME[zat.DOTA] = 'DoTA-CAP (dev+eval)'
try:
    reads = [a for row, by in READS.items() for s in (zat.DOTA, zat.DADA) for a in ('--read', s, row, str(by[s]))]
    zat.main([*reads, '--control', K5DFI, '--seed', str(SEED), '--name', NAME, '--out-dir', str(local)])
finally:
    zat.POOLING[zat.DOTA], zat.SET_NAME[zat.DOTA] = old_pool, old_name
md_path = local / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md'
ruler = json.loads((READS[L0]['dada'] / 'dada_readout.json').read_text())['middle_ruler_macro']
md_path.write_text(md_path.read_text().replace('DoTA-CAP-dev and DADA-2000 after', 'DoTA-CAP (dev+eval) and DADA-2000 after')
                   + f"\nDADA-2000 position only (no pixels): middle ruler macro {ruler['mean']:.4f} "
                     f"[{ruler['low']:.4f}, {ruler['high']:.4f}].\n")
publish(local, REPORTS)
print((REPORTS / f'{constants.ZS_ARM_TABLE_STEM}_{NAME}.md').read_text())
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste REPORTS/{constants.ZS_ARM_TABLE_STEM}_{NAME}.md to Claude')